# Faruq-v3 STB-SR1 — Seed-42 Kaggle Screening
Frozen Stage-A screen for `STB-SR1`: CMC-style classification representation + attention-only W-MSA/SW-MSA residual.

**Required Kaggle input:** existing private dataset `faruq-v3-experiment-core-v1`.

This notebook trains **seed 42 only**, evaluates grouped development `val`, and stops. Locked test, seeds 123/2026, AF2/WAV fusion, and post-result retuning are forbidden here.


In [ ]:
from pathlib import Path
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
if not INPUT.is_dir() or not WORK.is_dir(): raise RuntimeError('Notebook ini Kaggle-only.')
core=sorted(INPUT.rglob('af2_spectral_kaggle_manifest.json'))
if len(core)!=1: raise FileNotFoundError(f'Harus ada tepat satu core manifest; ditemukan {core}')
print('INPUT PREFLIGHT PASS')
print('CORE:',core[0])


In [ ]:
import importlib,json,os,shutil,subprocess,sys,time,torch
os.chdir(WORK)
REPO=WORK/'coffee-bean-detection'
BRANCH='agent/stb-sr1-selective-residual'
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(3):
    r=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
for m in list(sys.modules):
    if m=='coffee_detector' or m.startswith('coffee_detector.'): sys.modules.pop(m,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('COMMIT:',COMMIT)
print('ULTRALYTICS:',__import__('ultralytics').__version__)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan Kaggle GPU sebelum Run All.')


In [ ]:
from coffee_detector.experiments.prepare_af2_spectral_kaggle import prepare_af2_spectral_kaggle_input
DATA,ARTIFACTS,CORE_CONTRACT=prepare_af2_spectral_kaggle_input(INPUT,WORK)
assert CORE_CONTRACT['test_images_accessed'] is False
D0=Path(ARTIFACTS['D0_seed42_best.pt'])
GROUPED=DATA/'faruq_grouped_summary.json'
if not D0.is_file() or not GROUPED.is_file(): raise FileNotFoundError(f'Core artifact missing: D0={D0} GROUPED={GROUPED}')
print('CORE CONTRACT PASS')
print('DATA:',DATA)
print('D0:',D0)
print('GROUPED:',GROUPED)


In [ ]:
# Contract tests + static authorization before training.
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_stb_sr1.py'],cwd=REPO,check=True)
OUT=WORK/'stb-sr1-seed42-v1'; OUT.mkdir(exist_ok=True)
STATIC=OUT/'static_audit.json'
cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_stb_sr1',
     '--data-root',str(DATA),'--grouped-summary',str(GROUPED),'--d0-checkpoint',str(D0),
     '--output-root',str(OUT),'--stage','static','--seed','42']
subprocess.run(cmd,cwd=REPO,check=True)
audit=json.loads(STATIC.read_text(encoding='utf-8'))
print(json.dumps(audit,indent=2))
if audit['decision']!='PASS' or audit['test_opened'] is not False: raise RuntimeError('STOP: STB-SR1 static audit gagal.')
print('STATIC AUTHORIZATION PASS')


In [ ]:
# Train exactly one frozen arm: STB-SR1 seed 42.
RESULT=OUT/'val_reports'/'stb_sr1_seed42_decision.json'
LOG=OUT/'STBSR1_seed42.log'
if not RESULT.is_file():
    cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_stb_sr1',
         '--data-root',str(DATA),'--grouped-summary',str(GROUPED),'--d0-checkpoint',str(D0),
         '--output-root',str(OUT),'--stage','train','--seed','42','--device','0','--authorize-training']
    print('START STB-SR1 seed42',flush=True)
    with LOG.open('a',encoding='utf-8') as stream:
        p=subprocess.Popen(cmd,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
    previous=None
    while p.poll() is None:
        csv=OUT/'STBSR1_seed42'/'results.csv'
        epoch=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
        if epoch!=previous: print(f'STB-SR1: {epoch}/50 epoch',flush=True); previous=epoch
        time.sleep(120)
    if p.returncode:
        tail='\n'.join(LOG.read_text(errors='replace').splitlines()[-180:]) if LOG.is_file() else '<no log>'
        raise RuntimeError(f'STB-SR1 gagal, returncode={p.returncode}\n{tail}')
else:
    print('REUSE COMPLETE RESULT:',RESULT)
if not RESULT.is_file(): raise RuntimeError(f'Hasil tidak ditemukan: {RESULT}')
result=json.loads(RESULT.read_text(encoding='utf-8'))
assert result['evaluation_split']=='val' and result['test_opened'] is False and result['test_images_accessed'] is False
print('\n=== STB-SR1 SEED42 SCREEN ===')
for name,metrics in result['models'].items(): print(name,metrics)
print('\nDELTA VS CMC0:',result['comparison']['delta_vs_CMC0'])
print('DELTA VS STB1:',result['comparison']['delta_vs_STB1'])
print('\nCMC CRITERIA:',result['comparison']['cmc_improvement_criteria'])
print('STB RETENTION:',result['comparison']['stb_retention_criteria'])
print('ADVANCEMENT:',result['comparison']['stb_advancement_signals'])
print('\nDECISION:',result['decision'])
print('NEXT:',result['next_action'])
print('PARAM OVERHEAD VS CMC0:',result['parameter_overhead_vs_cmc0'])
print('PARAM OVERHEAD VS STB1:',result['parameter_overhead_vs_stb1'])
archive=Path(shutil.make_archive(str(WORK/'stb-sr1-seed42-output'),'zip',OUT))
print('\nFINAL ZIP:',archive)
print('RESULT:',RESULT)
print('STOP HERE. Jangan jalankan seed 123/2026 atau locked test sebelum hasil seed42 direview.')
